During analysis, I noticed to large issues with our data set:
* The Philippines event had an extra round
* The Malaysian event didn't supply nation textual information on the site



# Part 0

Don't run these first two code blocks here, or their output will be lost.

```
result = list(collection.find({
    "wins": {"$gt": 5}
}))
df= pd.DataFrame(result)
df.drop(columns=['_id'], inplace=True)
# Let's inspect our dataframe for posterity
df.nation.unique()
```

array(['Dark States', 'Dragon Empire', 'Keter Sanctuary', 'Brandt Gate',
       'Stoicheia', 'Buddyfight', 'Lyrical Monasterio', 'BanG Dream!', ''],
      dtype=object)

```
df.describe()
```


          rank 	        wins
count	362.000000	362.000000
mean	20.002762	7.022099
std	21.254493	1.318758
min	1.000000	6.000000
25%	6.000000	6.000000
50%	12.000000	7.000000
75%	24.000000	8.000000
max	91.000000	13.000000


The above are copied cells from the data analysis notebook, showing the issue of `''` in the list of nations, and the max win being 91.
The cells that isolated this issue were overwritten and covnerted into cells that solved the issue on a data frame, so for verification, at the end of this notebook, we should ensure that this issues no longer remain with the data when writing to the databases.

Now that we have our groundwork lined up, we still need to get our imports, and establish our DB connection.

In [ ]:

import src.db_operations
import src.helpers

import pandas as pd


# Part 1: Accounting the Extra Round
Since there was an extra round in the Philippines, it seemed most apporoprite to keep things in line with the other data, to reduce each player's win count by one for that event, and ensureing the value still 'min's out at 0.

In [2]:
# pipeline = [
# 	{'$group': {
# 		'_id': None,
# 		'total_wins': {'$sum': '$wins'},
# 		'avg_wins': {'$avg': '$wins'},
# 		'count': {'$sum': 1}
# 	}}
# ]

In [3]:
# main_collection.delete_many({"location": "0"})
# main_collection.delete_many({"location": "Philippines"})
# manila_collection = client['JSONproto']["manila"]
# new_manila = manila_collection.find({})
# main_collection.insert_many(new_manila)

In [4]:
# result = list(main_collection.find({
#     # "wins": {"$gt": 5}
# }))
# df= pd.DataFrame(result)
# df.drop(columns=['_id'], inplace=True)
# df.describe()
# df.nation.unique()

In [5]:
df_phil = db_operations.get_answer_from_table('main_table',
                                    'all_events',
                                    {'location':'Philippines'})

# Reduce each players wins by 1 to acoount for extra round
df_phil.loc[:, 'wins'] -= 1 
# Make sure players with negitive wins are set to 0
df_phil.loc[df_phil['wins'] == -1, 'wins'] = 0

df_fixed_phil = df_phil
df_fixed_phil.describe()

,rank,wins
count,441.00000,441.000000
mean,221.00000,2.575964
std,127.44999,2.179296
min,1.00000,0.000000
25%,111.00000,1.000000
50%,221.00000,2.000000
75%,331.00000,4.000000
max,441.00000,12.000000


# Part 2: Nation Mapping


In [6]:

df_mal = db_operations.get_answer_from_table('main_table',
                                    'all_events',
                                    {'location':'Malaysia'})

df_mal.boss.unique()

array(['DZ-BT09/SR08EN : Sugary and Scary Land, Heartluru',
       'DZ-BT09/004EN : Sugary and Scary Land, Heartluru',
       'DZ-SS09/001EN-R : Rewrite the Star, Vyrgilla',
       'DZ-BT07/001EN : Sealed Blaze of Arbitration, Bavsargra Aksayya',
       'DZ-SS04/001EN-R : Rewrite the Star, Vyrgilla',
       'DZ-BT07/FFR07EN : Keeper of the Moon Gate, Veissrugr',
       'BSF2023/VGD03EN : Fountain of Knowledge, Eva',
       'DZ-BT09/FFR04EN : Sugary and Scary Land, Heartluru',
       'D-BT08/FFR07EN : Omniscience Regalia, Minerva',
       'DZ-BT07/SR01EN : Sealed Blaze of Arbitration, Bavsargra Aksayya',
       'DZ-BT02/FR05EN : Rewrite the Star, Vyrgilla',
       'DZ-SS04/001EN : Rewrite the Star, Vyrgilla',
       'G-CHB03/Re:02EN : Vampire Princess of Night Fog, Nightrose',
       'DZ-TB01/002EN : Future Force Possessor, Tasuku Ryuenji',
       'D-BT08/007EN : Omniscience Regalia, Minerva',
       'D-SS05/Re43EN : Quaking Ripple Monster, Eledglema',
       'D-BT02/001EN : Dragonic Ov

In [7]:
df_mal.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date
0,69b03f963efa47b42b6c3545,1,Champion,"DZ-BT09/SR08EN : Sugary and Scary Land, Heartluru",10,,50Y33,{'RideDeck': {'DZ-BT09/SR08EN : Sugary and Sca...,Malaysia,AO,"December 6, 2025"
1,69b03f963efa47b42b6c3546,2,Champion,"DZ-BT09/004EN : Sugary and Scary Land, Heartluru",9,,5F15K,{'RideDeck': {'DZ-BT09/004EN : Sugary and Scar...,Malaysia,AO,"December 6, 2025"
2,69b03f963efa47b42b6c3547,3,Champion,"DZ-SS09/001EN-R : Rewrite the Star, Vyrgilla",9,,6HF8D,{'RideDeck': {'DZ-SS09/001EN-R : Rewrite the S...,Malaysia,AO,"December 6, 2025"
3,69b03f963efa47b42b6c3548,4,Ben Chan,"DZ-SS09/001EN-R : Rewrite the Star, Vyrgilla",8,,5AP89,{'RideDeck': {'DZ-SS09/001EN-R : Rewrite the S...,Malaysia,AO,"December 6, 2025"
4,69b03f963efa47b42b6c3549,5,Tai Luung,"DZ-BT09/004EN : Sugary and Scary Land, Heartluru",9,,4XZ8V,{'RideDeck': {'DZ-BT09/004EN : Sugary and Scar...,Malaysia,AO,"December 6, 2025"


In [8]:
df_mal.nation.unique()

array(['', 'BanG Dream!'], dtype=object)

In [9]:
nations = [
    "Brandt Gate",            # 0
    "Dark States",            # 1
    "Dragon Empire",          # 2
    "Keter Sanctuary",        # 3
    "Stoicheia",              # 4
    "Lyrical Monasterio",     # 5
    "BanG Dream!",            # 6
    "SHAMAN KING",            # 7
    "Buddyfight",             # 8
    "Touken Ranbu"            # 9
]

In [10]:
bosses = {
    'Sugary and Scary Land, Heartluru': nations[1],
    'Rewrite the Star, Vyrgilla': nations[2],
    'Sealed Blaze of Arbitration, Bavsargra Aksayya': nations[2],
    'Omniscience Regalia, Minerva': nations[3],
    'Vampire Princess of Night Fog, Nightrose': nations[4],
    'Future Force Possessor, Tasuku Ryuenji': nations[8],
    'Dragonic Overlord': nations[2],
    'Aurora Battle Princess, Seraph Snow': nations[0],
    'Destined King of Infinity, Levidras Empireo': nations[4],
    'Fated One of Zero, Blangdmire': nations[1],
    'Destined One of Scales, Aelquilibra': nations[2],
    'Fountain of Knowledge, Eva': nations[0],
    'Keeper of the Moon Gate, Veissrugr': nations[0],
    'Fated One of Unparalleled, Varga Dragres': nations[2],
    'Destined One of Time, Liael=Odium': nations[1],
    'Oblivionis the Noble': nations[6],
    'Quaking Ripple Monster, Eledglema': nations[0],
    'Demonic Jewel Dragon Emperor, Drajeweled Magnus': nations[1],
    'Galactic B-Hero, Stately Crozard': nations[0],
    'Fated One of Miracles, Rezael': nations[3],
    'Sylvan Horned Beast Emperor, Magnolia Patriarch': nations[4],
    "Fated One of Guiding Star, Welstra 'Blitz Arms'": nations[0],
    "Demon Stealth Dragon, Shiranui 'Oboro'": nations[2],
    'PolyPhonicOverDrive, Artisaria': nations[5],
    'Purgatory Dragon Deity, Favrneel': nations[1],
    'Sword of the Nation, Bastion Accord': nations[3],
    'Fated One of Ever-changing, Krysrain': nations[5],
    'Direful Doll Master, Androld': nations[1],
    'Savior of the World, Kyoya Gaen': nations[8],
    'Dragheart, Luard': nations[3],
    'Destined One of Exceedance, Impauldio': nations[0],
    'Absolute Zero, Sagitta': nations[5],
    'Crying Heart, Tomori Takamatsu': nations[6],
    'Argo Siren, Calliopeia': nations[4],
    "Youthberk 'Skyfall Arms: Rebuild'": nations[3],
    'Belgr the Skyruler': nations[4],
    'Pining Flower Maiden, Claudine': nations[4],
    'Incandescent Lion, Blond Ezel': nations[3],
    'Cardinal Dominus, Orfist Regis': nations[0],
    'Fated One of Taboo, Zorga Nadir': nations[4],
    'One Who Guides Towards Paradise, Nannaclir': nations[4],
    'Cloudwater Flitfoot Stealth Rogue, Shojodoji': nations[2],
    'Hound Raiser, Aoi': nations[3],
    'Chakrabarthi Phoenix Dragon, Nirvana Jheva': nations[2],
    'LèVre♡SœurS, Charmout': nations[5],
    'Sword Saint Knight Dragon, Gramgrace': nations[3],
    "Dragonic Kaiser Vermillion 'THE BLOOD'": nations[2],
    'Destined One of Supremacy, Lisciafael': nations[5],
    'Majesty Lord Blaster': nations[3],
    'Super Dimensional Robo, Daiyusha': nations[0],
    'Brightest Dreamer, Lilfa': nations[5],
    'Poison Knight of Silence Undercover, Mordalion': nations[3],
    'Chronojet Dragon': nations[1],
    'Avaricious Demonic Dragon, Greedon': nations[1],
    'Sword Saint Knight Dragon Emperor, Gramgrace Sieg': nations[3],
    'Grand March of Full Bloom, Lianorn': nations[4],
    "Blue Deathster, 'Skyrender' Avantgarda": nations[0],
    'Alter Ego Messiah': nations[0],
    'Sing with Me, Loronerol': nations[5],
    'FL∀MMe-Glam, Rougia': nations[5],
    'Scarlet Flame Marshal Dragon, Gandeeva': nations[2],
    'Juicy-Fruity, Tuarina': nations[5],
    'Give the Whole World a Spark, Tetsuya Kurodake': nations[8],
    'Legend of the Celestial Fury, Estacion': nations[4],
    'Darkness-exposing Demon,  Black': nations[0],
    "Aspiring 'Ninja-user of Justice', Zanya Kisaragi": nations[8],
    'Fiery Immolation Dragon, Kotihblaze': nations[2],
    'Venatiol, Dhinbarada': nations[2],
    'Astesice△Live, Kairi': nations[5],
    'Dragconnector, Grhyaundra': nations[2],
    'Venerable, Stoeirhaja': nations[4],
    'Astesice, Kairi': nations[5],
    'Malwyrm, Sybilt': nations[2],
}

In [ ]:

for i, row in df_mal.iterrows():
    boss_name = src.helpers.extract_card_name(row.boss)
    df_mal.loc[i, 'nation'] = bosses[boss_name]

df_fixed_mal = df_mal
df_fixed_mal.nation.unique()

array(['Dark States', 'Dragon Empire', 'Brandt Gate', 'Keter Sanctuary',
       'Stoicheia', 'Buddyfight', 'Lyrical Monasterio', 'BanG Dream!'],
      dtype=object)

In [12]:
df_fixed = pd.concat([df_fixed_phil, df_fixed_mal])
df_fixed.describe()
# df_fixed.location.unique()

,rank,wins
count,735.000000,735.000000
mean,192.485714,2.804082
std,118.267993,2.089218
min,1.000000,0.000000
25%,92.500000,1.000000
50%,185.000000,3.000000
75%,279.000000,4.000000
max,441.000000,12.000000


Looks like we got 'em all ^_^

# Part 3: Saving the Changes

TODO

I need to make a sort of 'transaction for the note book session, so I don't accidentially bork something up in develpment.
If the original database wasn't in memory still, I would have accidently droped the main collection, which would have problamatic to say the least.

In [13]:
# Ran into an issue with 'set on a copy of a slice'
# So I need to reframe how we do the df prep

# df_phil = pd.DataFrame(df[df['location'] == 'Philippines'])
# df_mal  = pd.DataFrame(df[df['location'] == 'Malaysia'])

We're saving both sets of changes to the main table, but we're only going to move the malaysian nation changes to the individual table, and not the win adjustment for the Philippines.

I'd like to perserve the original win data somehow, but the missing nation informaiton doesn't need preserving.

In [ ]:
# This update method was a bit cringe, but it works :/
from pymongo import UpdateOne
updates = []

# #Update the main table
for i, row in df_fixed_phil.iterrows():
    filter = {'_id': row['_id']}
    update = {'$set': {'wins': row['wins']}}
    updates.append(UpdateOne(filter, update))
    
for i, row in df_fixed_mal.iterrows():
    filter = {'_id': row['_id']}
    update = {'$set': {'nation': row['nation']}}
    updates.append(UpdateOne(filter, update))

src.db_operations.update_table('main_table', 'all_events',  updates)


# Update Malaysia Specific Table
filter = {}
payload = df_mal.to_dict(orient='records')
src.db_operations.overwrite_table('JSONproto', 'Malaysia', payload)


# Don't delete code, right?
# ## df_phil.drop(columns=['_id'], inplace=True)
# phil_collection = client["JSONproto"]['manila']
# phil_collection.drop()
# phil_collection.insert_many(df_phil.to_dict(orient='records'))


I ran into an issue where I didn't realize some of the code from the original creation was still present, so the philippines set was cut off for players with 5 or less wins.

I can fix this in a few easy steps thanks to the pipeline refurbishing. I'll just run it again on the Manila event.

Alright, the data was successfully returned to it's litte part of the data base, now we need to run this whole thing again, or at least the philippines bits

BOOM! We're all done! The data in the central database and the isolated ones have all been updated, despite the dificulties of deleting the data of players with less than 5 wins, but we got there :)